In [1]:
 !pip install kafka-python


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip uninstall kafka -y
!pip uninstall kafka-python -y


Found existing installation: kafka-python 2.2.15
Uninstalling kafka-python-2.2.15:
  Successfully uninstalled kafka-python-2.2.15


In [3]:
!pip install kafka-python
!pip install six

  Using cached kafka_python-2.2.15-py2.py3-none-any.whl.metadata (10.0 kB)
Using cached kafka_python-2.2.15-py2.py3-none-any.whl (309 kB)



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import json
import random
import uuid
import time
from datetime import datetime, timedelta
from kafka import KafkaProducer

EVENTHUBS_NAMESPACE = "ehhospital.servicebus.windows.net"
EVENT_HUB_NAME = "ehhospital"
CONNECTION_STRING = "Endpoint=sb://ehhospital.servicebus.windows.net/;SharedAccessKeyName=RootManageSharedAccessKey;SharedAccessKey=CZJAIx6ZB8E0n7xJRwTGF0iIxMLVfiV6Y+AEhD/aCls="

producer = KafkaProducer(
    bootstrap_servers=[f"{EVENTHUBS_NAMESPACE}:9093"],
    security_protocol="SASL_SSL",
    sasl_mechanism="PLAIN",
    sasl_plain_username="$ConnectionString",
    sasl_plain_password=CONNECTION_STRING,
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

departments = ["Emergency", "Surgery", "ICU", "Pediatrics", "Maternity", "Oncology", "Cardiology"]
genders = ["Male", "Female"]

def inject_dirty_data(record):
    if random.random() < 0.05:
        record["age"] = random.randint(101, 150)  

    if random.random() < 0.05:
        record["admission_time"] = (datetime.utcnow() + timedelta(hours=random.randint(1, 72))).isoformat()

    if random.random() < 0.2:
        record["discharge_time"] = None  

    return record

def generate_patient_event():
    age = random.choices(
        population=list(range(0, 101)),
        weights=[1]*5 + [5]*20 + [10]*51 + [5]*20 + [1]*5,
        k=1
    )[0]

    department = random.choices(
        population=departments,
        weights=[30, 15, 20, 10, 10, 5, 10], 
        k=1
    )[0]

    admission_time = datetime.utcnow() - timedelta(hours=random.randint(0, 72))
    discharge_time = admission_time + timedelta(hours=random.randint(24, 120))  

    event = {
        "patient_id": str(uuid.uuid4()),  
        "gender": random.choice(genders),  
        "age": age,   
        "department": department,
        "admission_time": admission_time.isoformat(),
        "discharge_time": discharge_time.isoformat(),
        "bed_id": random.randint(1, 500),
        "hospital_id": random.randint(1, 7)
    }

    return inject_dirty_data(event)

if __name__ == "__main__":
    while True:
        event = generate_patient_event()
        producer.send(EVENT_HUB_NAME, event)
        print(f"Sent to Event Hub: {json.dumps(event)}")
        time.sleep(random.randint(10, 30))


C:\Users\TABARAK\AppData\Local\Temp\ipykernel_29768\104392293.py:49: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  admission_time = datetime.utcnow() - timedelta(hours=random.randint(0, 72))


Sent to Event Hub: {"patient_id": "452a491e-33c1-422d-90f5-88a9fe352b8a", "gender": "Male", "age": 8, "department": "ICU", "admission_time": "2025-11-13T15:05:07.260707", "discharge_time": "2025-11-17T04:05:07.260707", "bed_id": 150, "hospital_id": 1}
Sent to Event Hub: {"patient_id": "305dd37a-e00f-48ee-9568-9d9b6edca16d", "gender": "Female", "age": 42, "department": "Pediatrics", "admission_time": "2025-11-14T00:05:18.621778", "discharge_time": null, "bed_id": 415, "hospital_id": 7}
Sent to Event Hub: {"patient_id": "e4a6b4dd-a2d0-49bc-a6d5-4d5aee150cbf", "gender": "Female", "age": 105, "department": "Pediatrics", "admission_time": "2025-11-12T10:05:42.624699", "discharge_time": "2025-11-14T03:05:42.624699", "bed_id": 252, "hospital_id": 4}


C:\Users\TABARAK\AppData\Local\Temp\ipykernel_29768\104392293.py:29: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record["admission_time"] = (datetime.utcnow() + timedelta(hours=random.randint(1, 72))).isoformat()


Sent to Event Hub: {"patient_id": "991f0c87-0c0d-45e7-bb39-4b10b54e0de3", "gender": "Female", "age": 19, "department": "Maternity", "admission_time": "2025-11-17T13:05:58.626619", "discharge_time": "2025-11-15T00:05:58.625869", "bed_id": 12, "hospital_id": 7}
Sent to Event Hub: {"patient_id": "7efe240d-7e36-4ce9-81dd-00ca517995c7", "gender": "Female", "age": 66, "department": "Cardiology", "admission_time": "2025-11-11T17:06:28.628634", "discharge_time": "2025-11-16T17:06:28.628634", "bed_id": 366, "hospital_id": 3}
Sent to Event Hub: {"patient_id": "9bdd4d8d-2298-48b4-8e66-7c4f4975567d", "gender": "Male", "age": 55, "department": "Emergency", "admission_time": "2025-11-13T06:06:47.629652", "discharge_time": "2025-11-17T12:06:47.629652", "bed_id": 198, "hospital_id": 2}
Sent to Event Hub: {"patient_id": "c9f86764-9ca4-4634-a4a9-e5b82d97aff5", "gender": "Female", "age": 117, "department": "Emergency", "admission_time": "2025-11-16T13:07:17.630810", "discharge_time": null, "bed_id": 391,